# Corrective RAG (CRAG): a simple, step-by-step example

**Plain RAG** searches the documents and hands **whatever it finds** to the LLM. If the search returns the wrong
chunks (wrong year, wrong leave type, nothing useful), the LLM still answers, often wrongly.

**Corrective RAG** adds a **quality check** between searching and answering. It looks at what the search returned and
**corrects** course when the results are not good enough.

```
 question
    │
    ▼
 1. RETRIEVE  ── hybrid search (BM25 keywords + Pinecone vectors, merged with RRF)
    │
    ▼
 2. GRADE     ── Gemini grades every chunk: relevant / partial / irrelevant
    │
    ▼
 3. DECIDE    ── CORRECT?   ──yes──────────────────────────────┐
    │                                                          │
    │ AMBIGUOUS or INCORRECT                                   │
    ▼                                                          │
    rewrite the question, search again (up to 2 times) ────────┤
    │                                                          │
    │ still nothing useful                                     │
    ▼                                                          ▼
    web search (optional) or "not found"          4. REFINE  ── keep only the evidence sentences
                                                               │
                                                               ▼
                                                  5. GENERATE ── answer with citations
```

**The three verdicts**

| Verdict | When | What we do |
|---|---|---|
| **CORRECT** | at least one chunk is **relevant** | answer from the good chunks |
| **AMBIGUOUS** | only **partial** matches (related, but not the answer) | rewrite the question, search again, merge the results |
| **INCORRECT** | **nothing** useful | rewrite and retry; if still nothing: web search (if allowed) or say *"not found"* |

**Our data:** the company leave policies in `../Leave_Policy`, one PDF per year (2015–2026). The numbers change every
year, so the right **year** matters a lot.

**Before you run:** the `.env` file must contain `PineCone_API_Key`, `Pinecone_Index_Name` and `GEMINI_API_KEY`.
Run the cells from top to bottom.

Steps 1–8 build the same hybrid search as `Hybrid_RAG.ipynb`. Steps 9–16 are the new **corrective** part.

### Step 1: Imports and settings

**What this cell does:** loads the libraries and sets the values used later.

| Library | Used for |
|---|---|
| `pypdf` | reading text from the PDFs |
| `RecursiveCharacterTextSplitter` | cutting text into small chunks |
| `BM25Okapi` (rank_bm25) | keyword search |
| `SentenceTransformer` | turning text into vectors for meaning search |
| `Pinecone` | the database that stores the vectors and searches them |
| `genai`, `types` | Google Gemini: grades chunks, rewrites questions and writes the answer |

**Settings**

| Setting | Meaning |
|---|---|
| `DATA_FOLDER` | where the policy files are. `..` = one folder up, because this notebook is inside `Corrective RAG/` |
| `SUPPORTED_EXTENSIONS` | which files we read (`.pdf`, `.txt`, `.md`). `manifest.json` is skipped |
| `NAMESPACE` | our own section inside the Pinecone index, so we don't mix with other notebooks' data |
| `CHUNK_SIZE`, `CHUNK_OVERLAP` | chunk size in characters, and how much neighbouring chunks share |
| `TOP_K` | how many chunks one search returns |
| `MAX_RETRIES` | how many times we may **rewrite the question** when the results are not good enough |
| `REINDEX` | `True` = always rebuild the Pinecone vectors. `False` = reuse them if they are already there |
| `ALLOW_WEB` | `True` = allow a Google web search as the very last resort |

**Output:** a confirmation line, or an error naming any API key missing from `.env`.

In [1]:
import os
import re
import json
import time
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

load_dotenv(find_dotenv(usecwd=True))

DATA_FOLDER = Path("../Leave_Policy")
SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md"}

NAMESPACE = "corrective-rag"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # 384-dim vectors
LLM_MODEL = "gemini-3.5-flash"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
TOP_K = 6          # chunks retrieved per search
MAX_RETRIES = 2    # question rewrites when retrieval is not good enough

REINDEX = False    # True = rebuild the Pinecone namespace
ALLOW_WEB = False  # True = allow web search as a last resort

missing = [k for k in ("PineCone_API_Key", "Pinecone_Index_Name", "GEMINI_API_KEY") if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing in .env: {', '.join(missing)}")

print("Settings loaded. Data folder:", DATA_FOLDER.resolve())

c:\Users\DELL\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Settings loaded. Data folder: C:\Users\Public\Documents\Python Training\Leave_Policy


### Step 2: Read the documents

Reading is split into three small cells: **2.1** read one file, **2.2** find its year and document id,
**2.3** read the whole folder.

#### 2.1 Read one file

**What `read_file(path)` does:** returns the text of one file as a list of `(page_number, text)`.
- **PDF:** reads every page with `page.extract_text()`. `or ""` handles pages with no text (e.g. a scanned image).
- **.txt / .md:** the whole file is treated as one page (page 1).

**Output:** nothing (used in 2.3).

In [2]:
def read_file(path):
    """Return a list of (page_number, text) for one file."""
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(path)
        return [(n, page.extract_text() or "") for n, page in enumerate(reader.pages, start=1)]
    return [(1, path.read_text(encoding="utf-8", errors="ignore"))]

#### 2.2 Find the policy year and document id

Each policy applies to **one year**, so every chunk must know its year.

**`detect_year(path, text)`**
1. First looks in the **file name**: `Leave_Policy_2019.pdf` → `2019`. The pattern `(?:19|20)\d{2}` means
   "19 or 20 followed by two digits".
2. If the name has no year, looks in the **text** for `Policy year 2019`.
3. Returns `None` if neither is found.

**`detect_doc_id(text)`** finds the document code, e.g. `HR-LV-2019` (`\d{4}` = four digits).

**Output:** the year and id detected from an example file name and text.

In [3]:
def detect_year(path, text):
    """Policy year from the file name (Leave_Policy_2019.pdf), else from the text."""
    match = re.search(r"(?:19|20)\d{2}", path.stem)
    if match:
        return int(match.group())
    match = re.search(r"Policy year\s*((?:19|20)\d{2})", text)
    return int(match.group(1)) if match else None


def detect_doc_id(text):
    """Document code such as HR-LV-2019, or None."""
    match = re.search(r"HR-LV-\d{4}", text)
    return match.group(0) if match else None


print(detect_year(Path("Leave_Policy_2019.pdf"), ""))        # year from the file name
print(detect_year(Path("policy.txt"), "Policy year 2021 ..."))  # year from the text
print(detect_doc_id("Document HR-LV-2019, effective 1 Jan"))

2019
2021
HR-LV-2019


#### 2.3 Read every document in the folder

**What `load_documents(folder)` does:**
1. Goes through every file in the folder (sorted by name).
2. Skips files that are not `.pdf`, `.txt` or `.md` (e.g. `manifest.json`).
3. Reads the file (2.1) and finds its year and document id (2.2). If there is no `HR-LV-...` code, the file name is used as the id.
4. Saves every non-empty page with its **file**, **year**, **doc_id** and **page number**, so later we can say where an
   answer came from.

**Output:** the skipped files, how many pages were read, and the start of the first page.

In [4]:
def load_documents(folder):
    pages = []
    for path in sorted(p for p in folder.iterdir() if p.is_file()):
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            print(f"  skipping {path.name} (not a policy document)")
            continue

        file_pages = read_file(path)
        full_text = "\n".join(text for _, text in file_pages)
        year = detect_year(path, full_text)
        doc_id = detect_doc_id(full_text) or path.stem

        for page_no, text in file_pages:
            if text.strip():
                pages.append({"file": path.name, "year": year, "doc_id": doc_id,
                              "page": page_no, "text": text})

    print(f"Read {len(pages)} pages from {len({p['file'] for p in pages})} files")
    return pages


pages = load_documents(DATA_FOLDER)
print("\nFirst page starts with:\n", pages[0]["text"][:300])

  skipping manifest.json (not a policy document)
Read 47 pages from 12 files

First page starts with:
 Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources 


### Step 3: Split the text into chunks

**What `make_chunks(pages)` does:** cuts every page into small pieces (**chunks**) of about 800 characters
(`CHUNK_SIZE`), where neighbouring chunks share up to 100 characters (`CHUNK_OVERLAP`) so a sentence cut at the
border isn't lost.

Each chunk is a dictionary:

| Key | Example | Used for |
|---|---|---|
| `id` | `Leave_Policy_2019-p2-c1` | file 2019, page 2, chunk 1. Links Pinecone results back to the chunk |
| `year`, `doc_id`, `page` | `2019`, `HR-LV-2019`, `2` | showing where an answer came from |
| `label` | `Leave Policy 2019 (HR-LV-2019), page 2` | printed in the logs and the sources list |
| `text` | `Leave Policy 2019 (HR-LV-2019), page 2: Employees receive ...` | what is searched and graded |

**Important trick:** the label is put **in front of the text**. A chunk from the middle of a page might not mention its
year; with the label, both searches and the grader always know which year a chunk belongs to.

`chunk_by_id` lets us look up a chunk by its id.

**Output:** the number of chunks and one example.

In [5]:
def make_chunks(pages):
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = []
    for p in pages:
        for i, piece in enumerate(splitter.split_text(p["text"])):
            label = f"Leave Policy {p['year']} ({p['doc_id']}), page {p['page']}"
            chunks.append({
                "id": f"{Path(p['file']).stem}-p{p['page']}-c{i}",
                "year": p["year"],
                "doc_id": p["doc_id"],
                "page": p["page"],
                "label": label,
                "text": f"{label}: {piece}",   # label in the text, so both searches see the year
            })
    return chunks


chunks = make_chunks(pages)
chunk_by_id = {c["id"]: c for c in chunks}

print(f"Created {len(chunks)} chunks\n")
print("Example chunk:", chunks[0]["id"])
print(chunks[0]["text"][:400])

Created 147 chunks

Example chunk: Leave_Policy_2015-p1-c0
Leave Policy 2015 (HR-LV-2015), page 1: Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources Department
Applies to
All permanent employees of Acme Techno


### Step 4: Build the keyword index (BM25)

**1. `tokenize(text)`: split text into search words**
- `text.lower()`: lowercase, so *Sick* and *sick* match.
- `re.findall(r"[a-z0-9]+", ...)`: keep only letters and digits. `HR-LV-2019` becomes `hr`, `lv`, `2019`.
- Remove **stopwords** (*the, of, what, how, many ...*): they appear everywhere and don't help.

Example: *"How many days of Earned Leave in 2019?"* → `['days', 'earned', 'leave', '2019']`

**2. `BM25Okapi(...)`: the keyword index.** BM25 gives every chunk a score for a question: more matching words →
higher score, and **rare** words (like *2019*) count more than common ones (like *leave*).

**Output:** the words of an example question.

In [6]:
STOPWORDS = {"a", "an", "the", "of", "in", "on", "for", "to", "and", "or", "is", "are", "was",
             "were", "what", "which", "how", "many", "much", "does", "do", "did", "can", "i",
             "my", "me", "it", "this", "that", "by", "with", "at", "as", "from", "per", "get",
             "got"}


def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOPWORDS]


bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

print(tokenize("How many days of Earned Leave did employees get in 2019?"))

['days', 'earned', 'leave', 'employees', '2019']


### Step 5: Build the meaning index (embeddings in Pinecone)

Meaning search needs every chunk turned into a **vector** (a list of 384 numbers that represents its meaning) and
stored in Pinecone. Texts with similar meaning get similar numbers, e.g. *"look after a sick parent"* ≈ *"Caregiver leave"*.

| Cell | Job |
|---|---|
| 5.1 | load the embedding model |
| 5.2 | connect to Pinecone |
| 5.3 | count the vectors already stored |
| 5.4 | upload the vectors, **only if needed** |

#### 5.1 Load the embedding model

**What this cell does:** loads `all-MiniLM-L6-v2`, a small free model that runs **on your computer** (no API key).
It is downloaded the first time (about 90 MB). Every text becomes a vector of **384 numbers**.

**Output:** the vector length (384) and the first 5 numbers of an example.

In [7]:
model = SentenceTransformer(EMBED_MODEL)

example = model.encode("Caregiver leave")
print("Vector length:", len(example))
print("First 5 numbers:", example[:5])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1136.42it/s]


Vector length: 384
First 5 numbers: [-0.01955347 -0.01959092  0.0015571   0.02956124  0.04665591]


#### 5.2 Connect to Pinecone

**What this cell does:**
1. Logs in with `PineCone_API_Key` and reads the index name from `Pinecone_Index_Name`.
2. Creates the index **only if it doesn't exist yet** (`dimension=384` must match the model; `metric="cosine"`
   compares vectors by their angle).
3. `pc.Index(...)` gives us `index`, which we use to upload and search.

**Output:** the index name.

In [8]:
pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))
index_name = os.getenv("Pinecone_Index_Name")

if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=384, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index = pc.Index(index_name)
print("Connected to Pinecone index:", index_name)

Connected to Pinecone index: agenticai-training


#### 5.3 Count the vectors already stored

**What `count_vectors()` does:** asks Pinecone how many vectors are in **our namespace** (`corrective-rag`).
Returns 0 if the namespace doesn't exist yet.

We use it to **avoid re-uploading** every run: if Pinecone already holds exactly as many vectors as we have chunks,
the index is up to date.

**Output:** the number stored now and the number of chunks.

In [9]:
def count_vectors():
    ns = index.describe_index_stats().namespaces.get(NAMESPACE)
    return ns.vector_count if ns else 0


print(f"Vectors in Pinecone: {count_vectors()}   Chunks: {len(chunks)}")

Vectors in Pinecone: 147   Chunks: 147


#### 5.4 Upload the vectors (only if needed)

**What `sync_index(reindex)` does:**
1. **Skips** everything if the namespace already has exactly `len(chunks)` vectors and `reindex` is `False`.
2. Otherwise **clears** the old vectors in our namespace (other notebooks' data is not touched).
   The first time the namespace doesn't exist; `try / except` ignores that error.
3. Turns **all** chunks into vectors in one call (`model.encode`).
4. Builds one **record** per chunk: `id` (the chunk id), `values` (the 384 numbers), `metadata` (year and page).
5. Uploads them in groups of 100 with `upsert` (= insert, or replace if the id already exists).
6. **Waits** until Pinecone has saved them: Pinecone needs a few seconds before new vectors can be searched.
   Checks every 2 seconds, up to 60 seconds.

Set `REINDEX = True` in Step 1 if you changed the documents but the chunk count stayed the same.

**Output:** "up to date", or how many vectors were stored.

In [10]:
def sync_index(reindex=False):
    if not reindex and count_vectors() == len(chunks):
        print(f"Pinecone namespace '{NAMESPACE}' is up to date ({len(chunks)} vectors)")
        return

    try:
        index.delete(delete_all=True, namespace=NAMESPACE)
    except Exception:
        pass  # namespace does not exist yet (first run)

    print(f"Embedding and uploading {len(chunks)} chunks to Pinecone...")
    vectors = model.encode([c["text"] for c in chunks])
    records = [{"id": c["id"], "values": vectors[i].tolist(),
                "metadata": {"year": c["year"] or 0, "page": c["page"]}}
               for i, c in enumerate(chunks)]

    for start in range(0, len(records), 100):
        index.upsert(vectors=records[start:start + 100], namespace=NAMESPACE)

    for _ in range(30):  # Pinecone is eventually consistent
        if count_vectors() == len(chunks):
            break
        time.sleep(2)
    print(f"Stored {count_vectors()} of {len(chunks)} vectors")


sync_index(reindex=REINDEX)

Pinecone namespace 'corrective-rag' is up to date (147 vectors)


### Step 6: Keyword search

**What `keyword_search(question, k)` does:**
1. Splits the question into words with `tokenize` (Step 4).
2. `bm25.get_scores(...)` gives **every** chunk a score.
3. Sorts from highest to lowest and returns the **ids** of the top `k` (default 20).

**Output:** the top 3 chunk ids for a test question.

In [11]:
def keyword_search(question, k=20):
    scores = bm25.get_scores(tokenize(question))
    ranked = sorted(zip(chunks, scores), key=lambda pair: pair[1], reverse=True)
    return [c["id"] for c, _score in ranked[:k]]


print(keyword_search("earned leave in 2019")[:3])

['Leave_Policy_2019-p2-c2', 'Leave_Policy_2019-p4-c0', 'Leave_Policy_2019-p1-c0']


### Step 7: Meaning search

**What `vector_search(question, k)` does:**
1. Turns the question into a vector with the **same** model used for the chunks.
2. Asks Pinecone for the `k` (default 20) most similar vectors in our namespace.
3. Returns their ids, keeping only ids we know (`chunk_by_id`), in case old vectors are left in Pinecone.

**Output:** the top 3 chunk ids for the same test question. Compare the years with Step 6.

In [12]:
def vector_search(question, k=20):
    result = index.query(vector=model.encode(question).tolist(), top_k=k, namespace=NAMESPACE)
    return [m.id for m in result.matches if m.id in chunk_by_id]


print(vector_search("earned leave in 2019")[:3])

['Leave_Policy_2019-p2-c2', 'Leave_Policy_2018-p2-c2', 'Leave_Policy_2019-p1-c2']


### Step 8: Hybrid search (Reciprocal Rank Fusion)

**What `hybrid_search(question, top_k)` does:** merges the keyword list and the meaning list into **one** ranking.

The two kinds of score can't be added (keyword ≈ 0–30, meaning ≈ 0–1), so **RRF** uses only each chunk's
**position** in each list:

```
RRF score = 1 / (60 + rank in keyword list)  +  1 / (60 + rank in meaning list)
```

A chunk that **both** searches rank high gets two contributions and rises to the top.
Returns the top `TOP_K` (6) **chunk dictionaries**. This is step **1. RETRIEVE** of Corrective RAG.

**Output:** the labels of the chunks found for a test question.

In [13]:
def hybrid_search(question, top_k=TOP_K):
    rrf = {}
    for ranked_ids in (keyword_search(question), vector_search(question)):
        for rank, chunk_id in enumerate(ranked_ids, start=1):
            rrf[chunk_id] = rrf.get(chunk_id, 0) + 1 / (60 + rank)
    best = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    return [chunk_by_id[i] for i in best]


for c in hybrid_search("How many days of earned leave did employees get in 2019?"):
    print(c["label"])

Leave Policy 2019 (HR-LV-2019), page 2
Leave Policy 2019 (HR-LV-2019), page 1
Leave Policy 2019 (HR-LV-2019), page 3
Leave Policy 2019 (HR-LV-2019), page 3
Leave Policy 2019 (HR-LV-2019), page 2
Leave Policy 2016 (HR-LV-2016), page 2


---
## The corrective part

Everything above is ordinary Hybrid RAG. From here on, Gemini is used not only to **answer**, but also to
**grade** the search results and **rewrite** the question when they are poor.

### Step 9: Connect to Google Gemini

**What `ask_gemini(prompt, as_json, tools)` does:** sends a prompt to Gemini and returns the reply text.

| Option | Meaning |
|---|---|
| `temperature=0` | no randomness: the same prompt gives (almost) the same reply. Good for grading |
| `as_json=True` | asks Gemini to reply in **JSON** (used by the grader in Step 11) |
| `tools=[...]` | lets Gemini use a tool, here **Google Search** (used for the web fallback in Step 14) |

If you see **503 UNAVAILABLE**, Google's servers are busy: wait a minute and run the cell again,
or change `LLM_MODEL` in Step 1 (e.g. to `gemini-3.5-flash-lite`).

**Output:** nothing.

In [14]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


def ask_gemini(prompt, as_json=False, tools=None):
    config = types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json" if as_json else None,
        tools=tools,
    )
    response = client.models.generate_content(model=LLM_MODEL, contents=prompt, config=config)
    return response.text or ""

### Step 10: Read Gemini's JSON safely

**Problem:** we ask Gemini for JSON, but sometimes it wraps it in extra text or code fences, e.g.
````
Here are the grades:
```json
[{"id": 1, "grade": "relevant"}]
```
````
`json.loads` would fail on that and crash the program.

**What `parse_json(text, default)` does:**
1. Tries `json.loads(text)` directly.
2. If that fails, cuts out the part from the first `[` or `{` to the last `]` or `}` (`re.DOTALL` lets `.` match
   new lines too) and tries again.
3. If that also fails, returns `default` instead of crashing (for the grader, an empty list = "nothing graded").

**Output:** three examples: clean JSON, JSON with extra text, and broken text.

In [15]:
def parse_json(text, default):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"[\[{].*[\]}]", text, re.DOTALL)  # strip stray prose / fences
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    return default


print(parse_json('[{"id": 1}]', default=[]))
print(parse_json('Here you go:\n```json\n[{"id": 1}]\n```', default=[]))
print(parse_json("sorry, no JSON here", default=[]))

[{'id': 1}]
[{'id': 1}]
[]


### Step 11: Grade the retrieved chunks (step 2. GRADE)

This is the **core of Corrective RAG**: before using the chunks, Gemini checks each one.

**What `grade_chunks(question, chunks)` does:**
1. Numbers the chunks `[1]`, `[2]` ... and puts them all in **one** prompt (one Gemini call instead of six).
2. Asks Gemini to return, for every chunk:
   - `grade`: **relevant** (contains the answer, for the right year), **partial** (related but not the answer),
     or **irrelevant**
   - `evidence`: the exact sentence(s) that help answer the question
3. The prompt stresses the **year**: a chunk from the wrong year is *not* relevant when the question names a year.
4. Reads the JSON with `parse_json` (Step 10) and matches each grade back to its chunk by number
   (`by_number.get(n) or by_number.get(str(n))` handles Gemini returning `1` or `"1"`).
5. Anything missing or unexpected counts as **irrelevant**, the safe choice.
6. Returns **copies** of the chunks with `grade` and `evidence` added (`{**c, ...}` copies a dictionary), so the
   original chunks are not changed.

**Output:** the grade of each chunk for a test question, plus one piece of evidence.

In [16]:
def grade_chunks(question, chunks):
    """Grade all chunks in one call. Returns copies with 'grade' and 'evidence' added."""
    listing = "\n\n".join(f"[{n}] {c['text']}" for n, c in enumerate(chunks, start=1))
    prompt = f"""You grade search results for a question about company leave policies.
Each policy applies to one calendar year only. A chunk from the wrong year is NOT relevant
when the question names a year. If the question names no year, the latest year is preferred.

For every chunk return:
  "id": the chunk number,
  "grade": "relevant"   - it directly contains the answer, for the right year
           "partial"    - related topic or useful background, but does not answer it
           "irrelevant" - does not help,
  "evidence": the exact sentence(s) from the chunk that help answer the question
              ("" when irrelevant).

Return a JSON list only.

Question: {question}

Chunks:
{listing}"""
    grades = parse_json(ask_gemini(prompt, as_json=True), default=[])
    by_number = {g.get("id"): g for g in grades if isinstance(g, dict)}

    graded = []
    for n, c in enumerate(chunks, start=1):
        g = by_number.get(n) or by_number.get(str(n)) or {}
        grade = str(g.get("grade", "irrelevant")).lower()
        graded.append({**c,
                       "grade": grade if grade in {"relevant", "partial"} else "irrelevant",
                       "evidence": (g.get("evidence") or "").strip()})
    return graded


test_question = "How many days of earned leave did employees get in 2019?"
test_graded = grade_chunks(test_question, hybrid_search(test_question))
for c in test_graded:
    print(f"{c['grade']:<10}  {c['label']}")

best = next((c for c in test_graded if c["grade"] == "relevant"), None)
if best:
    print("\nEvidence from", best["label"], ":\n ", best["evidence"])

relevant    Leave Policy 2019 (HR-LV-2019), page 2
relevant    Leave Policy 2019 (HR-LV-2019), page 1
irrelevant  Leave Policy 2019 (HR-LV-2019), page 3
irrelevant  Leave Policy 2019 (HR-LV-2019), page 3
irrelevant  Leave Policy 2019 (HR-LV-2019), page 2
irrelevant  Leave Policy 2016 (HR-LV-2016), page 2

Evidence from Leave Policy 2019 (HR-LV-2019), page 2 :
  Earned Leave (also called Privilege Leave) entitlement in 2019 is 18 days, accruing at a monthly rate of 1/12 of the annual entitlement.


### Step 12: Decide the verdict (step 3. DECIDE)

**What `decide(graded)` does:** turns the grades into one verdict.

| If ... | Verdict |
|---|---|
| any chunk is **relevant** | `CORRECT` |
| otherwise, any chunk is **partial** | `AMBIGUOUS` |
| otherwise | `INCORRECT` |

**Output:** the verdict for the chunks graded in Step 11, and for three hand-made examples.

In [17]:
def decide(graded):
    if any(c["grade"] == "relevant" for c in graded):
        return "CORRECT"
    if any(c["grade"] == "partial" for c in graded):
        return "AMBIGUOUS"
    return "INCORRECT"


print("Step 11 chunks:", decide(test_graded))
print(decide([{"grade": "partial"}, {"grade": "relevant"}]))    # CORRECT
print(decide([{"grade": "partial"}, {"grade": "irrelevant"}]))  # AMBIGUOUS
print(decide([{"grade": "irrelevant"}]))                        # INCORRECT

Step 11 chunks: CORRECT
CORRECT
AMBIGUOUS
INCORRECT


### Step 13: Rewrite the question (the correction)

When the verdict is `AMBIGUOUS` or `INCORRECT`, the user's wording may simply not match the policy's wording.
E.g. *"time off when my dad is in hospital"* vs. the policy term *"Caregiver leave"*.

**What `rewrite_question(question, attempt, previous)` does:** asks Gemini for a better **search query**:
- use the formal policy terms for the leave type
- keep any year mentioned
- drop filler words
- be **different** from the earlier queries (`previous`), so a retry doesn't repeat the same failed search

`.strip().strip('"')` removes spaces and quotes around the reply. If Gemini returns nothing, the original question is kept.

**Output:** a rewritten version of an informal question.

In [ ]:
def rewrite_question(question, attempt, previous):
    prompt = f"""The search below did not find a good answer in the company leave policy
documents (one PDF per year, 2015-2026, with leave types such as casual, sick, earned,
maternity, paternity, caregiver, wellness, sabbatical and so on).

Rewrite the question as a better search query: use the formal policy terms for the leave
type, keep any year mentioned, and drop filler words. Give a different query from the
earlier attempts. Return only the query text.

Original question: {question}
Earlier queries: {previous}
Attempt: {attempt}"""
    return ask_gemini(prompt).strip().strip('"') or question


q = "can I get days off when my dad is in hospital?"
print("Original :", q)
print("Rewritten:", rewrite_question(q, 1, [q]))

### Step 14: Merge results from several searches

After a rewrite we search again. The new chunks are **added** to the ones we already have (the **pool**), so nothing
good from an earlier search is lost.

**What `merge(kept, new)` does:**
- Puts the kept chunks in a dictionary by id.
- Adds each new chunk. If the same chunk was found **twice**, it keeps the copy with the **better** grade
  (`relevant` 2 > `partial` 1 > `irrelevant` 0), because the second search may have been graded differently.

**Output:** a hand-made example where chunk `A` is upgraded from partial to relevant.

In [ ]:
def merge(kept, new):
    """Add newly graded chunks, keeping the best grade when a chunk appears twice."""
    order = {"relevant": 2, "partial": 1, "irrelevant": 0}
    by_id = {c["id"]: c for c in kept}
    for c in new:
        if c["id"] not in by_id or order[c["grade"]] > order[by_id[c["id"]]["grade"]]:
            by_id[c["id"]] = c
    return list(by_id.values())


first = [{"id": "A", "grade": "partial"}, {"id": "B", "grade": "irrelevant"}]
second = [{"id": "A", "grade": "relevant"}, {"id": "C", "grade": "partial"}]
print(merge(first, second))

### Step 15: Web search fallback (optional)

**What `web_search_answer(question)` does:** the **last resort** when the policies contain nothing useful, even after
rewriting. It lets Gemini use **Google Search** (`types.GoogleSearch()`) and asks for a short general answer that
clearly says it is **not the company's policy**.

It is only used when `ALLOW_WEB = True` (Step 1). With `False`, the pipeline answers *"not found"* instead, which is
the safer choice for HR questions.

**Output:** nothing (used in Step 17).

In [ ]:
def web_search_answer(question):
    """Last resort: answer from Google Search via Gemini grounding."""
    prompt = f"""The company's own leave policy documents do not cover this question.
Give a short general answer from the web and say clearly that it is general information,
not the company's policy.

Question: {question}"""
    return ask_gemini(prompt, tools=[types.Tool(google_search=types.GoogleSearch())])

### Step 16: Generate the answer (steps 4. REFINE and 5. GENERATE)

**What `generate_answer(question, evidence_chunks)` does:**
1. **Refine:** for each chunk it uses only the **evidence sentences** found by the grader (Step 11), not the whole
   chunk. This removes unrelated text that could confuse Gemini. If a chunk has no evidence, its full text is used.
2. Labels each one `[Source 1]`, `[Source 2]` ... with its year and page.
3. Asks Gemini to answer **only** from these sources, use the **year** the question asks about (or the latest year
   if none), cite its sources, and say so if the answer isn't there.

**Output:** an answer built from the chunks graded in Step 11.

In [ ]:
def generate_answer(question, evidence_chunks):
    context = "\n\n".join(
        f"[Source {n}] {c['label']}:\n{c['evidence'] or c['text']}"
        for n, c in enumerate(evidence_chunks, start=1)
    )
    prompt = f"""Answer the question using only the sources below.
These are leave policies for different years; use the policy of the year the question asks
about. If no year is asked, use the latest year in the sources and say which year.
Cite the sources you used, e.g. [Source 2]. If the sources do not contain the answer, say so.

{context}

Question: {question}"""
    return ask_gemini(prompt)


useful = [c for c in test_graded if c["grade"] != "irrelevant"]
print(generate_answer(test_question, useful))

### Step 17: The full Corrective RAG pipeline

**What `corrective_rag(question)` does:** puts Steps 8–16 together.

**The retry loop** (at most `MAX_RETRIES + 1` = 3 searches):
1. Search with the latest query (`hybrid_search`).
2. Grade the chunks **against the original question** (`grade_chunks`). The rewritten query is only used for
   *searching*; relevance is always judged against what the user actually asked.
3. Add them to the pool (`merge`) and decide the verdict on the **whole pool** (`decide`).
4. `CORRECT` → stop. Otherwise, rewrite the question (`rewrite_question`) and loop.

**After the loop:**
- Collect the **relevant** and **partial** chunks as evidence, sorted: relevant first, then newest year first.
- **No evidence at all** → web search (if `allow_web`) or *"I could not find this ..."*.
- **Some evidence** → `generate_answer`.

**Returns** a dictionary: `question`, `verdict`, `source` (`documents`, `web` or `none`), `answer` and `sources`.

With `verbose=True` it prints every search, every grade and every verdict, so you can watch it correct itself.

**Output:** nothing (used below).

In [ ]:
def corrective_rag(question, allow_web=ALLOW_WEB, verbose=True):
    def log(msg):
        if verbose:
            print(msg)

    log(f"QUESTION: {question}")
    queries = [question]
    pool = []

    # 1-3. Retrieve, grade, decide; rewrite and retry if needed
    for attempt in range(MAX_RETRIES + 1):
        query = queries[-1]
        log(f"\n  Retrieval #{attempt + 1}  query: {query!r}")
        graded = grade_chunks(question, hybrid_search(query))
        for c in graded:
            log(f"    {c['grade']:<10}  {c['label']}")
        pool = merge(pool, graded)

        verdict = decide(pool)
        log(f"  Verdict: {verdict}")
        if verdict == "CORRECT" or attempt == MAX_RETRIES:
            break
        queries.append(rewrite_question(question, attempt + 1, queries))  # AMBIGUOUS / INCORRECT

    # 4. Keep the useful chunks: relevant first, newest year first
    relevant = [c for c in pool if c["grade"] == "relevant"]
    partial = [c for c in pool if c["grade"] == "partial"]
    evidence = sorted(relevant + partial,
                      key=lambda c: (c["grade"] != "relevant", -(c["year"] or 0)))

    # Nothing useful: web search or "not found"
    if not evidence:
        if allow_web:
            log("  Nothing useful in the policies -> web search fallback")
            return {"question": question, "verdict": "INCORRECT", "source": "web",
                    "answer": web_search_answer(question), "sources": []}
        return {"question": question, "verdict": "INCORRECT", "source": "none",
                "answer": "I could not find this in the Leave Policy documents (2015-2026).",
                "sources": []}

    # 5. Generate the answer from the evidence
    return {"question": question, "verdict": decide(pool), "source": "documents",
            "answer": generate_answer(question, evidence),
            "sources": [f"{c['label']} ({c['grade']})" for c in evidence]}

### Step 18: Print a result nicely

**What `print_result(result)` does:** prints the answer, then the numbered sources (with their grade), or a note when
the answer came from the web.

**Output:** nothing (used below).

In [ ]:
def print_result(result):
    print("\nANSWER:", result["answer"].strip())
    if result["sources"]:
        print("\nSOURCES:")
        for n, s in enumerate(result["sources"], start=1):
            print(f"  [Source {n}] {s}")
    elif result["source"] == "web":
        print("\n(Answer from web search, not from the company policies.)")
    print("-" * 80)

### Step 19: Try it

Watch the log of each question: the grades, the verdict, and whether the question was rewritten.

**Question 1: a clear question with a year.** The first search should already find a relevant 2019 chunk, so the
verdict is **CORRECT** after one retrieval. Expected: **18 days**.

In [ ]:
print_result(corrective_rag("How many days of earned leave did employees get in 2019?"))

**Question 2: informal wording.** The policy calls it *Caregiver leave*. If the first search only finds related chunks
(**AMBIGUOUS**), the question is rewritten with the policy term and searched again.

In [ ]:
print_result(corrective_rag("Can I get days off when my dad is in hospital?"))

**Question 3: not in the policies.** The leave policies don't cover salaries, so every search should come back
**INCORRECT**. After the retries, the pipeline says *"not found"* instead of inventing an answer.
This is the main benefit of Corrective RAG over plain RAG.

In [ ]:
print_result(corrective_rag("What is the salary of a senior software engineer?"))

**Question 4: the same question with the web fallback.** With `allow_web=True`, a question the policies can't answer
gets a general answer from Google Search, clearly marked as **not** the company's policy.

In [ ]:
print_result(corrective_rag("What is the salary of a senior software engineer?", allow_web=True))

**Your turn:** ask your own question. Set `verbose=False` to hide the log and see only the answer.

In [ ]:
print_result(corrective_rag("When was fertility treatment leave introduced and how many days does it give?"))